In [94]:
import pandas as pd
import importlib
import funciones as f #Tener funciones.py en mismo directorio. Tiene las funciones usadas para procesar un df
importlib.reload(f)

data = pd.read_csv('competition_data.csv')
submission = pd.read_csv('submission.csv')
submission_aux = pd.read_csv('submission.csv')

### Descomentar la siguiente celda la primera vez que se corre el notebook

In [95]:
# uri_to_ms_data = f.get_songs_durations(data)
# uri_to_ms_submission = f.get_songs_durations(submission)
# diccionario = {**uri_to_ms_data, **uri_to_ms_submission}

In [96]:
# Ejemplo de uso de procesar_df
submission = f.procesar_df(submission, diccionario)

c:\Users\dafyd\Documents\Escuela\2025\semestre 1\TD6\TP2\funciones.py:122: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  suma = df['reason_fwdbtn'].rolling(ventana).sum()
c:\Users\dafyd\Documents\Escuela\2025\semestre 1\TD6\TP2\funciones.py:123: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  conteo = df['reason_fwdbtn'].rolling(ventana).count()


In [97]:
submission

,ts,Unnamed: 0,platform,master_metadata_track_name,master_metadata_album_artist_name,master_metadata_album_album_name,spotify_track_uri,shuffle,hour,day_of_week,...,fwdbtn_seguidos,trackdone_seguidos,fwdbtn_spree,trackdone_spree,fwdbtn_prop_30,duration_ms,track_prop,artist_prop,album_prop,hour day_of_week
0,2014-06-27 18:01:15+00:00,74916,"iOS 7.0.4 (iPod5,1)",Mejor,Los Tipitos,Push,spotify:track:5LFl6vXC2CwcciAbymL4jZ,False,18,4,...,0,0,0,0,0.000000,232493.0,0.000040,0.001837,0.000519,72.0
1,2014-09-04 21:46:57+00:00,74923,"iOS 7.0.4 (iPod5,1)","Circles - Based On Ludovico Einaudi ""Experience""",Ludovico Einaudi,In a Time Lapse,spotify:track:0mEsOEi4rWBy0IXE5oTKr2,False,21,3,...,1,0,0,0,1.000000,237500.0,0.000040,0.000240,0.000040,63.0
2,2014-09-04 21:48:51+00:00,74924,"iOS 7.0.4 (iPod5,1)",Primavera,Ludovico Einaudi,Divenire,spotify:track:0fzw4BBD5FRJtPuQbUUKzJ,False,21,3,...,0,0,0,0,0.500000,445746.0,0.000040,0.000240,0.000200,63.0
3,2016-06-23 21:07:59+00:00,74933,OS X 10.11.5 [x86 4],NaN,NaN,NaN,NaN,False,21,3,...,0,0,0,0,0.000000,NaN,NaN,NaN,NaN,63.0
4,2016-06-23 21:08:03+00:00,74934,OS X 10.11.5 [x86 4],NaN,NaN,NaN,NaN,False,21,3,...,0,0,0,0,0.000000,NaN,NaN,NaN,NaN,63.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25032,2024-05-22 15:28:48+00:00,74898,ios,Zafar,La Vela Puerca,A Contraluz,spotify:track:1wIUWGdTdhVk5gIPd0ULxX,True,15,2,...,0,1,0,0,0.000000,262000.0,0.000919,0.001598,0.001518,30.0
25033,2024-05-22 15:35:04+00:00,74899,ios,Un Loco En La Calesita,Juan Carlos Baglietto,Baglietto,spotify:track:3mHOEGxXbUpk5CZDgQhrUP,True,15,2,...,1,0,0,0,0.333333,376426.0,0.000639,0.002117,0.001558,30.0
25034,2024-05-22 15:39:44+00:00,74900,ios,Dulce condena - Edición Aniversario,Los Rodriguez,Sin Documentos,spotify:track:4Pk1N5mY14kO5N3JcADgb2,True,15,2,...,0,1,0,0,0.333333,280920.0,0.000719,0.006151,0.001238,30.0
25035,2024-05-22 15:39:49+00:00,74901,ios,Yo No Quiero Volverme Tan Loco,Charly García,Pubis Angelical / Yendo De La Cama Al Living,spotify:track:68LeIVjVDRMXPlfdFHhID6,True,15,2,...,0,2,0,1,0.250000,310706.0,0.001358,0.021288,0.002836,30.0


In [98]:
# Ordenar data cronológicamente
data = f.sort_by_ts(data)

In [99]:
# KFold
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score
import numpy as np
import xgboost as xgb

def temporal_stratified_kfold(data, n_splits=5):
    '''
    Requiere: data esta ordenada cronologicamente y tiene una columna 'ts' con la fecha.
    Devuelve: los indices de entrenamiento y validación para cada fold en un KFold estratificado temporalmente.
    '''
    data['ts'] = pd.to_datetime(data['ts'])
    data['year'] = data['ts'].dt.year - 2000
    folds = []
    for fold in range(n_splits):
        idxs_train = []
        idxs_val = []
        for year, df_year in data.groupby('year'):
            df_year = df_year.sort_values('ts')

            fold_size = len(df_year) // n_splits
            val_start = fold * fold_size
            val_end   = (fold + 1) * fold_size if fold < n_splits - 1 else len(df_year)

            # .iloc aquí selecciona posiciones locales,
            # pero .index te devuelve los labels globales
            val_idx   = df_year.iloc[val_start:val_end].index
            train_idx = df_year.drop(val_idx).index

            idxs_train.extend(train_idx)
            idxs_val.extend(val_idx)
        folds.append((idxs_train, idxs_val))
    return folds

In [ ]:
kf = temporal_stratified_kfold(data, n_splits=5)

auc_scores = []
columns_to_drop = ['Unnamed: 0', 'ts','TARGET', 'master_metadata_track_name', 'master_metadata_album_artist_name', 'master_metadata_album_album_name', 'spotify_track_uri', 'platform']

for fold, tupla in enumerate(kf):
    train_idx, val_idx = tupla
    print(f"Fold {fold + 1}")

    train_fold = data.loc[train_idx].copy().reset_index(drop=True)
    val_fold = data.loc[val_idx].copy().reset_index(drop=True)

    # Aplicar pipeline modular a cada fold
    train_fold = f.procesar_df(train_fold, diccionario)
    val_fold = f.procesar_df(val_fold, diccionario)
    print('len trainfold',len(train_fold.columns))
    print('len valfold',len(val_fold.columns))

    # Entrenar el modelo
    clf_xgb = xgb.XGBClassifier(objective = 'binary:logistic',
                            seed = 42,
                            eval_metric = 'auc',
                            early_stopping_rounds = 100)
    clf_xgb.fit(train_fold.drop(columns=columns_to_drop),
                train_fold['TARGET'],
                eval_set=[(val_fold.drop(columns=columns_to_drop), val_fold['TARGET'])],
                verbose=False
                )

    # Evaluar sobre validación
    preds = clf_xgb.predict_proba(val_fold.drop(columns=columns_to_drop))[:, 1]
    auc = roc_auc_score(val_fold['TARGET'], preds)
    auc_scores.append(auc)
    print(f"AUC Fold {fold + 1}: {auc:.4f}")

# Resultado final
print(f"\nAUC promedio: {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")

Fold 1


c:\Users\dafyd\Documents\Escuela\2025\semestre 1\TD6\TP2\funciones.py:122: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  suma = df['reason_fwdbtn'].rolling(ventana).sum()
c:\Users\dafyd\Documents\Escuela\2025\semestre 1\TD6\TP2\funciones.py:123: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  conteo = df['reason_fwdbtn'].rolling(ventana).count()
c:\Users\dafyd\Documents\Escuela\2025\semestre 1\TD6\TP2\funciones.py:122: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  suma = df['reason_fwdbtn'].rolling(ventana).sum()
c:\Users\dafyd\Documents\Escuela\2025\semestre 1\TD6\TP2\funciones.py:123: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  conteo = df['reason_fwdbtn'].rolling(ventana).count()


len trainfold 33
len valfold 33
AUC Fold 1: 0.8588
Fold 2


c:\Users\dafyd\Documents\Escuela\2025\semestre 1\TD6\TP2\funciones.py:122: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  suma = df['reason_fwdbtn'].rolling(ventana).sum()
c:\Users\dafyd\Documents\Escuela\2025\semestre 1\TD6\TP2\funciones.py:123: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  conteo = df['reason_fwdbtn'].rolling(ventana).count()
c:\Users\dafyd\Documents\Escuela\2025\semestre 1\TD6\TP2\funciones.py:122: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  suma = df['reason_fwdbtn'].rolling(ventana).sum()
c:\Users\dafyd\Documents\Escuela\2025\semestre 1\TD6\TP2\funciones.py:123: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  conteo = df['reason_fwdbtn'].rolling(ventana).count()


len trainfold 33
len valfold 33
AUC Fold 2: 0.9111
Fold 3


...

len trainfold 33
len valfold 33
AUC Fold 3: 0.9027
Fold 4


c:\Users\dafyd\Documents\Escuela\2025\semestre 1\TD6\TP2\funciones.py:122: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  suma = df['reason_fwdbtn'].rolling(ventana).sum()
c:\Users\dafyd\Documents\Escuela\2025\semestre 1\TD6\TP2\funciones.py:123: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  conteo = df['reason_fwdbtn'].rolling(ventana).count()
c:\Users\dafyd\Documents\Escuela\2025\semestre 1\TD6\TP2\funciones.py:122: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  suma = df['reason_fwdbtn'].rolling(ventana).sum()
c:\Users\dafyd\Documents\Escuela\2025\semestre 1\TD6\TP2\funciones.py:123: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  conteo = df['reason_fwdbtn'].rolling(ventana).count()


len trainfold 33
len valfold 33
AUC Fold 4: 0.9005
Fold 5


c:\Users\dafyd\Documents\Escuela\2025\semestre 1\TD6\TP2\funciones.py:122: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  suma = df['reason_fwdbtn'].rolling(ventana).sum()
c:\Users\dafyd\Documents\Escuela\2025\semestre 1\TD6\TP2\funciones.py:123: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  conteo = df['reason_fwdbtn'].rolling(ventana).count()
c:\Users\dafyd\Documents\Escuela\2025\semestre 1\TD6\TP2\funciones.py:122: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  suma = df['reason_fwdbtn'].rolling(ventana).sum()
c:\Users\dafyd\Documents\Escuela\2025\semestre 1\TD6\TP2\funciones.py:123: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  conteo = df['reason_fwdbtn'].rolling(ventana).count()


len trainfold 33
len valfold 33
AUC Fold 5: 0.9190

AUC promedio: 0.8984 ± 0.0209
